In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [3]:
# =========================================================
# 프로젝트 및 데이터 경로
# =========================================================

# 현재 07번 노트북이 GitHub 프로젝트 폴더 내부에 있다고 가정
PROJECT_DIR = Path.cwd()

# notebooks 폴더 안에서 실행 중이면 프로젝트 루트로 이동
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

# GitHub 프로젝트 폴더와 같은 상위 폴더에 있는 project_data
DATA_DIR = PROJECT_DIR.parent / "project_data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
API_TEST_DIR = DATA_DIR / "api_test"

API_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("현재 작업 위치 :", Path.cwd())
print("프로젝트 위치   :", PROJECT_DIR)
print("데이터 위치     :", DATA_DIR)
print("processed 존재 :", PROCESSED_DIR.exists())
print("raw 존재       :", RAW_DIR.exists())
print("reference 존재 :", REFERENCE_DIR.exists())

현재 작업 위치 : /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks
프로젝트 위치   : /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project
데이터 위치     : /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data
processed 존재 : True
raw 존재       : True
reference 존재 : True


In [4]:
print("[processed 파일]")
for path in sorted(PROCESSED_DIR.glob("*")):
    print(path.name)

print("\n[raw 파일]")
for path in sorted(RAW_DIR.glob("*")):
    print(path.name)

print("\n[reference 파일]")
for path in sorted(REFERENCE_DIR.glob("*")):
    print(path.name)

[processed 파일]
all_age_commute_morning_0700_0940.csv
all_age_commute_od_aggregated.csv
all_age_commute_od_selected_70.csv
all_age_commute_od_selected_80.csv
all_age_commute_od_selected_90.csv
commute_destination_threshold_comparison.csv
commute_destination_threshold_summary.csv
youth_mobility_seoul_cleaned.csv

[raw 파일]
.DS_Store
10월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
11월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
12월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
1월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
2월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
3월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
4월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
5월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
6월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv

In [5]:
# =========================================================
# 1. 06번 주요 출근 목적지 결과 불러오기
# =========================================================

INPUT_FILE = (
    PROCESSED_DIR
    / "all_age_commute_od_selected_80.csv"
)

print("입력 파일:", INPUT_FILE)
print("입력 파일 존재:", INPUT_FILE.exists())

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"06번 결과 파일을 찾지 못했습니다: {INPUT_FILE}"
    )

입력 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_selected_80.csv
입력 파일 존재: True


In [6]:
# =========================================================
# 2. 80% 기준 주요 OD 불러오기
# =========================================================

selected_od = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    dtype={
        "거주동 코드": "string",
        "거주동 이름": "string",
        "근무동 코드": "string",
        "근무동 이름": "string",
    },
)

print("전체 OD 수:", f"{len(selected_od):,}")
print(
    "거주동 수:",
    f"{selected_od['거주동 코드'].nunique():,}"
)
print(
    "근무동 수:",
    f"{selected_od['근무동 코드'].nunique():,}"
)

display(selected_od.head())

전체 OD 수: 30,839
거주동 수: 428
근무동 수: 428


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,선택목적지_출근량합,최종_가중치,평균_이동시간_분,평균_이동거리_m,평균_이동거리_km
0,11110515,청운효자동,11110530,사직동,1,"55,710.2500","564,412.0400",0.0987,0.0987,"451,971.5100",0.1233,17.6800,928.2000,0.9280
1,11110515,청운효자동,11110615,종로1.2.3.4가동,2,"47,373.5200","564,412.0400",0.0839,0.1826,"451,971.5100",0.1048,25.7200,"1,759.8300",1.7600
2,11110515,청운효자동,11110515,청운효자동,3,"43,228.9200","564,412.0400",0.0766,0.2592,"451,971.5100",0.0956,15.4600,502.6900,0.5030
3,11110515,청운효자동,11140550,명동,4,"21,721.1700","564,412.0400",0.0385,0.2977,"451,971.5100",0.0481,30.4600,"2,213.6200",2.2140
4,11110515,청운효자동,11140520,소공동,5,"17,247.3800","564,412.0400",0.0306,0.3283,"451,971.5100",0.0382,30.1900,"2,127.8800",2.1280


In [7]:
# =========================================================
# 3. 기본 데이터 점검
# =========================================================

required_cols = [
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "목적지_순위",
    "출근_이동량",
    "목적지_출근비중",
    "누적_출근비중",
    "최종_가중치",
]

missing_cols = [
    col
    for col in required_cols
    if col not in selected_od.columns
]

if missing_cols:
    raise ValueError(
        f"필요한 컬럼이 없습니다: {missing_cols}"
    )


# 필수 값 결측 확인
missing_check = (
    selected_od[required_cols]
    .isna()
    .sum()
    .to_frame("결측치 수")
)

display(missing_check)


# OD 코드 중복 확인
duplicate_count = selected_od.duplicated(
    subset=[
        "거주동 코드",
        "근무동 코드",
    ]
).sum()

print("중복 OD 수:", f"{duplicate_count:,}")

assert duplicate_count == 0, (
    "같은 거주동-근무동 조합이 중복되어 있습니다."
)

,결측치 수
거주동 코드,0
거주동 이름,0
근무동 코드,0
근무동 이름,0
목적지_순위,0
출근_이동량,0
목적지_출근비중,0
누적_출근비중,0
최종_가중치,0


중복 OD 수: 0


In [8]:
# =========================================================
# 4. 06번 선택 결과 검증
# =========================================================

origin_check = (
    selected_od
    .groupby(
        [
            "거주동 코드",
            "거주동 이름",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        선택_OD수=("근무동 코드", "size"),
        최종_누적비중=("누적_출근비중", "max"),
        최종_가중치합=("최종_가중치", "sum"),
        선택_출근량=("출근_이동량", "sum"),
    )
)


origin_check["가중치합_오차"] = (
    origin_check["최종_가중치합"] - 1
).abs()


print("거주동 수:", f"{len(origin_check):,}")

print("\n최종 누적 비중")
display(
    origin_check["최종_누적비중"]
    .describe()
)

print("\n최종 가중치 합")
display(
    origin_check["최종_가중치합"]
    .describe()
)


invalid_weight = origin_check.loc[
    origin_check["가중치합_오차"] > 1e-6
]

print(
    "가중치 합이 1과 다른 거주동 수:",
    f"{len(invalid_weight):,}"
)

거주동 수: 428

최종 누적 비중


count   428.0000
mean      0.8015
std       0.0009
min       0.8000
25%       0.8008
50%       0.8015
75%       0.8022
max       0.8055
Name: 최종_누적비중, dtype: float64


최종 가중치 합


count   428.0000
mean      1.0000
std       0.0000
min       1.0000
25%       1.0000
50%       1.0000
75%       1.0000
max       1.0000
Name: 최종_가중치합, dtype: float64

가중치 합이 1과 다른 거주동 수: 0


In [9]:
# =========================================================
# 5. 내부 이동과 외부 이동 구분
# =========================================================

selected_od["내부이동_여부"] = (
    selected_od["거주동 코드"]
    == selected_od["근무동 코드"]
)


external_od = selected_od.loc[
    ~selected_od["내부이동_여부"]
].copy()


internal_od = selected_od.loc[
    selected_od["내부이동_여부"]
].copy()


print(
    "전체 주요 OD:",
    f"{len(selected_od):,}"
)

print(
    "외부 이동 OD:",
    f"{len(external_od):,}"
)

print(
    "내부 이동 OD:",
    f"{len(internal_od):,}"
)


total_flow = selected_od["출근_이동량"].sum()

internal_flow = internal_od["출근_이동량"].sum()

external_flow = external_od["출근_이동량"].sum()


print(
    "\n내부 이동량 비중:",
    f"{internal_flow / total_flow:.2%}"
)

print(
    "외부 이동량 비중:",
    f"{external_flow / total_flow:.2%}"
)

전체 주요 OD: 30,839
외부 이동 OD: 30,411
내부 이동 OD: 428

내부 이동량 비중: 9.60%
외부 이동량 비중: 90.40%


In [10]:
# =========================================================
# 6. 외부 이동 API 호출 대상 정리
# =========================================================

external_od["OD_ID"] = (
    external_od["거주동 코드"].astype("string")
    + "_"
    + external_od["근무동 코드"].astype("string")
)


# OD ID 중복 확인
duplicate_od_id = external_od["OD_ID"].duplicated().sum()

print("중복 OD ID 수:", f"{duplicate_od_id:,}")

assert duplicate_od_id == 0, (
    "OD_ID가 중복되어 있습니다."
)


# 내부 이동에도 식별자 생성
internal_od["OD_ID"] = (
    internal_od["거주동 코드"].astype("string")
    + "_"
    + internal_od["근무동 코드"].astype("string")
)


display(
    external_od[
        [
            "OD_ID",
            "거주동 코드",
            "거주동 이름",
            "근무동 코드",
            "근무동 이름",
            "목적지_순위",
            "출근_이동량",
            "최종_가중치",
        ]
    ].head()
)

중복 OD ID 수: 0


,OD_ID,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,최종_가중치
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,"55,710.2500",0.1233
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,"47,373.5200",0.1048
3,11110515_11140550,11110515,청운효자동,11140550,명동,4,"21,721.1700",0.0481
4,11110515_11140520,11110515,청운효자동,11140520,소공동,5,"17,247.3800",0.0382
5,11110515_11560540,11110515,청운효자동,11560540,여의동,6,"17,222.8300",0.0381


In [11]:
# =========================================================
# 7. 주민센터 좌표 검색 대상 행정동 생성
# =========================================================

origin_dong = (
    external_od[
        [
            "거주동 코드",
            "거주동 이름",
        ]
    ]
    .rename(
        columns={
            "거주동 코드": "행정동 코드",
            "거주동 이름": "행정동 이름",
        }
    )
    .copy()
)


destination_dong = (
    external_od[
        [
            "근무동 코드",
            "근무동 이름",
        ]
    ]
    .rename(
        columns={
            "근무동 코드": "행정동 코드",
            "근무동 이름": "행정동 이름",
        }
    )
    .copy()
)


dong_geocode_targets = (
    pd.concat(
        [
            origin_dong,
            destination_dong,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["행정동 코드"]
    )
    .sort_values(
        "행정동 코드"
    )
    .reset_index(drop=True)
)


# 08번 지오코딩 단계에서 사용할 검색어
dong_geocode_targets["주민센터_검색어"] = (
    "서울특별시 "
    + dong_geocode_targets["행정동 이름"]
    + " 주민센터"
)


# 08번에서 채울 좌표 결과 컬럼
dong_geocode_targets["대표_장소명"] = pd.NA
dong_geocode_targets["대표_주소"] = pd.NA
dong_geocode_targets["대표_위도"] = pd.NA
dong_geocode_targets["대표_경도"] = pd.NA
dong_geocode_targets["지오코딩_API"] = pd.NA
dong_geocode_targets["지오코딩_상태"] = "미호출"
dong_geocode_targets["지오코딩_오류"] = pd.NA


print(
    "좌표 검색 대상 행정동 수:",
    f"{len(dong_geocode_targets):,}"
)

print(
    "행정동 코드 중복:",
    dong_geocode_targets["행정동 코드"]
    .duplicated()
    .sum()
)

display(dong_geocode_targets.head(20))

좌표 검색 대상 행정동 수: 428
행정동 코드 중복: 0


,행정동 코드,행정동 이름,주민센터_검색어,대표_장소명,대표_주소,대표_위도,대표_경도,지오코딩_API,지오코딩_상태,지오코딩_오류
0,11110515,청운효자동,서울특별시 청운효자동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
1,11110530,사직동,서울특별시 사직동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
2,11110540,삼청동,서울특별시 삼청동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
3,11110550,부암동,서울특별시 부암동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
4,11110560,평창동,서울특별시 평창동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
5,11110570,무악동,서울특별시 무악동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
6,11110580,교남동,서울특별시 교남동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
7,11110600,가회동,서울특별시 가회동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
8,11110615,종로1.2.3.4가동,서울특별시 종로1.2.3.4가동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
9,11110630,종로5.6가동,서울특별시 종로5.6가동 주민센터,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>


In [12]:
# =========================================================
# 8. 경로 API 호출 결과 컬럼 준비
# =========================================================

api_targets = external_od.copy()


result_cols = [
    "출발지_위도",
    "출발지_경도",
    "도착지_위도",
    "도착지_경도",
    "대중교통_시간_분",
    "대중교통_거리_m",
    "대중교통_요금_원",
    "환승_횟수",
    "도보_시간_분",
    "도보_거리_m",
    "경로_API",
    "API_호출상태",
    "API_오류메시지",
]


for col in result_cols:
    api_targets[col] = pd.NA


api_targets["API_호출상태"] = "미호출"


print(
    "최종 API 호출 대상:",
    f"{len(api_targets):,}"
)

display(
    api_targets[
        [
            "OD_ID",
            "거주동 코드",
            "거주동 이름",
            "근무동 코드",
            "근무동 이름",
            "출근_이동량",
            "최종_가중치",
            "API_호출상태",
        ]
    ].head()
)

최종 API 호출 대상: 30,411


,OD_ID,거주동 코드,거주동 이름,근무동 코드,근무동 이름,출근_이동량,최종_가중치,API_호출상태
0,11110515_11110530,11110515,청운효자동,11110530,사직동,"55,710.2500",0.1233,미호출
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,"47,373.5200",0.1048,미호출
3,11110515_11140550,11110515,청운효자동,11140550,명동,"21,721.1700",0.0481,미호출
4,11110515_11140520,11110515,청운효자동,11140520,소공동,"17,247.3800",0.0382,미호출
5,11110515_11560540,11110515,청운효자동,11560540,여의동,"17,222.8300",0.0381,미호출


In [13]:
# =========================================================
# 9. 저장 컬럼 순서 정리
# =========================================================

api_front_cols = [
    "OD_ID",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "목적지_순위",
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "평균_이동시간_분",
    "평균_이동거리_m",
    "내부이동_여부",
]


api_result_cols = [
    "출발지_위도",
    "출발지_경도",
    "도착지_위도",
    "도착지_경도",
    "대중교통_시간_분",
    "대중교통_거리_m",
    "대중교통_요금_원",
    "환승_횟수",
    "도보_시간_분",
    "도보_거리_m",
    "경로_API",
    "API_호출상태",
    "API_오류메시지",
]


# 실제 데이터에 존재하는 컬럼만 선택
api_front_cols = [
    col
    for col in api_front_cols
    if col in api_targets.columns
]


api_targets = api_targets[
    api_front_cols
    + api_result_cols
].copy()

In [14]:
# =========================================================
# 10. 07번 결과 저장
# =========================================================

API_TARGET_FILE = (
    PROCESSED_DIR
    / "commute_api_targets_80.csv"
)

INTERNAL_OD_FILE = (
    PROCESSED_DIR
    / "commute_internal_od_80.csv"
)

GEOCODE_TARGET_FILE = (
    PROCESSED_DIR
    / "commute_dong_geocode_targets.csv"
)

API_TARGET_SUMMARY_FILE = (
    PROCESSED_DIR
    / "commute_api_target_summary.csv"
)

In [15]:
# =========================================================
# 11. 최종 결과 확인
# =========================================================

print("=" * 60)
print("07번 API 호출 대상 생성 결과")
print("=" * 60)

print(
    "API 호출 대상 OD 수:",
    f"{len(api_targets):,}"
)

print(
    "내부 이동 OD 수:",
    f"{len(internal_od):,}"
)

print(
    "좌표 검색 대상 행정동 수:",
    f"{len(dong_geocode_targets):,}"
)

print(
    "API 호출 대상 거주동 수:",
    f"{api_targets['거주동 코드'].nunique():,}"
)

print(
    "API 호출 대상 근무동 수:",
    f"{api_targets['근무동 코드'].nunique():,}"
)

display(api_targets.head(20))

07번 API 호출 대상 생성 결과
API 호출 대상 OD 수: 30,411
내부 이동 OD 수: 428
좌표 검색 대상 행정동 수: 428
API 호출 대상 거주동 수: 428
API 호출 대상 근무동 수: 428


,OD_ID,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,선택목적지_출근량합,최종_가중치,평균_이동시간_분,평균_이동거리_m,내부이동_여부,출발지_위도,출발지_경도,도착지_위도,도착지_경도,대중교통_시간_분,대중교통_거리_m,대중교통_요금_원,환승_횟수,도보_시간_분,도보_거리_m,경로_API,API_호출상태,API_오류메시지
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,"55,710.2500","564,412.0400",0.0987,0.0987,"451,971.5100",0.1233,17.6800,928.2000,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,"47,373.5200","564,412.0400",0.0839,0.1826,"451,971.5100",0.1048,25.7200,"1,759.8300",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
3,11110515_11140550,11110515,청운효자동,11140550,명동,4,"21,721.1700","564,412.0400",0.0385,0.2977,"451,971.5100",0.0481,30.4600,"2,213.6200",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
4,11110515_11140520,11110515,청운효자동,11140520,소공동,5,"17,247.3800","564,412.0400",0.0306,0.3283,"451,971.5100",0.0382,30.1900,"2,127.8800",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
5,11110515_11560540,11110515,청운효자동,11560540,여의동,6,"17,222.8300","564,412.0400",0.0305,0.3588,"451,971.5100",0.0381,40.9600,"7,382.1000",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
6,11110515_11680640,11110515,청운효자동,11680640,역삼1동,7,"13,614.6600","564,412.0400",0.0241,0.3829,"451,971.5100",0.0301,57.3300,"10,844.2900",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
7,11110515_11140540,11110515,청운효자동,11140540,회현동,8,"13,283.2400","564,412.0400",0.0235,0.4064,"451,971.5100",0.0294,32.2300,"2,790.2700",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
8,11110515_11110600,11110515,청운효자동,11110600,가회동,9,"11,631.6900","564,412.0400",0.0206,0.4271,"451,971.5100",0.0257,23.6700,"1,752.4900",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
9,11110515_11110640,11110515,청운효자동,11110640,이화동,10,"11,112.0000","564,412.0400",0.0197,0.4467,"451,971.5100",0.0246,37.8000,"3,151.9100",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>
10,11110515_11170625,11110515,청운효자동,11170625,한강로동,11,"8,745.8000","564,412.0400",0.0155,0.4622,"451,971.5100",0.0194,44.4200,"5,766.3300",False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,미호출,<NA>


In [16]:
# =========================================================
# 12. 07번 결과 파일 저장
# =========================================================

# 외부 이동 경로 API 호출 대상
api_targets.to_csv(
    API_TARGET_FILE,
    index=False,
    encoding="utf-8-sig",
)

# 내부 이동 OD
internal_od.to_csv(
    INTERNAL_OD_FILE,
    index=False,
    encoding="utf-8-sig",
)

# 행정동 주민센터 좌표 검색 대상
dong_geocode_targets.to_csv(
    GEOCODE_TARGET_FILE,
    index=False,
    encoding="utf-8-sig",
)

# 요약 정보
api_target_summary = pd.DataFrame(
    {
        "항목": [
            "전체 선택 OD 수",
            "외부 이동 API 대상 OD 수",
            "내부 이동 OD 수",
            "좌표 검색 대상 행정동 수",
            "API 대상 거주동 수",
            "API 대상 근무동 수",
        ],
        "값": [
            len(selected_od),
            len(api_targets),
            len(internal_od),
            len(dong_geocode_targets),
            api_targets["거주동 코드"].nunique(),
            api_targets["근무동 코드"].nunique(),
        ],
    }
)

api_target_summary.to_csv(
    API_TARGET_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("[저장 완료]")
print(API_TARGET_FILE)
print(INTERNAL_OD_FILE)
print(GEOCODE_TARGET_FILE)
print(API_TARGET_SUMMARY_FILE)

[저장 완료]
/Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_api_targets_80.csv
/Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_internal_od_80.csv
/Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_dong_geocode_targets.csv
/Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_api_target_summary.csv
